In [2]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

  Using cached pandas-3.0.3-cp312-cp312-win_arm64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_arm64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_arm64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-win_arm64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached plotnine-0.15.4-py3-none-any.whl.metadata (9.6 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached quarto-0.1.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached kagglehub-1.0.1-py3-none-any.whl.metadata (40 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_arm64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3

  error: subprocess-exited-with-error
  
  × Building wheel for statsmodels (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [1557 lines of output]
      C:\Users\chaou\AppData\Local\Temp\pip-build-env-6xz8a0kk\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: BSD License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-arm64-cpytho

# 🧠 Étape 5 : Modélisation (Machine Learning & Deep Learning) (Squelette Étudiant)

Cette étape correspond au cinquième chapitre du cours. L'objectif est d'implémenter d'une part un modèle de Machine Learning tabulaire (ex: RandomForest) et d'autre part un réseau de neurones convolutif (CNN) sous TensorFlow pour traiter des images ou signaux complexes.

### 1. Préparation de l'environnement

On importe toutes les bibliothèques nécessaires pour ce notebook :
- **pandas / numpy** : manipulation des données
- **matplotlib / seaborn** : visualisations
- **scikit-learn** : modèle Random Forest, encodage, split train/test
- **tensorflow** (optionnel) : réseau de neurones convolutif (CNN)

In [1]:

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.append(os.path.abspath('..'))

print("Librairies de modélisation importées avec succès !")


ModuleNotFoundError: No module named 'pandas'

### 2. Modélisation Tabulaire (Machine Learning) — Random Forest

Le **Random Forest** (forêt aléatoire) est un algorithme d'apprentissage supervisé qui combine plusieurs arbres de décision pour faire des prédictions plus robustes.

**Étapes :**
1. Chargement du dataset nettoyé (`cleaned_data_sample.csv`)
2. Conversion des colonnes numériques et encodage des variables catégorielles (marque, type, OS, gamme CPU)
3. Sélection de 17 features pertinentes pour prédire le prix
4. Découpage du jeu de données : **80% entraînement / 20% test**
5. Entraînement du modèle avec 100 arbres de décision
6. Visualisation de l'importance de chaque variable

> L'évaluation des métriques (MAE, RMSE, R²) est réalisée dans le notebook **06_evaluation.ipynb**.

In [ ]:

# Chargement des données nettoyées
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Conversion des colonnes numériques
df["Inches"] = pd.to_numeric(df["Inches"], errors="coerce")
df["Res_Width"] = pd.to_numeric(df["Res_Width"], errors="coerce")
df["Res_Height"] = pd.to_numeric(df["Res_Height"], errors="coerce")
df["Cpu_Frequence_GHz"] = pd.to_numeric(df["Cpu_Frequence_GHz"], errors="coerce")

# Encodage des variables catégorielles
le = LabelEncoder()
df["Company_enc"] = le.fit_transform(df["Company"].astype(str))
df["TypeName_enc"] = le.fit_transform(df["TypeName"].astype(str))
df["OpSys_enc"] = le.fit_transform(df["OpSys"].astype(str))
df["Cpu_Gamme_enc"] = le.fit_transform(df["Cpu_Gamme"].astype(str))

# Sélection des features
features = [
    "Inches", "Ram", "Cpu_Frequence_GHz", "weight_kg",
    "total_memory_gb", "Res_Width", "Res_Height",
    "has_ssd", "has_hdd", "has_intel_gpu", "has_nvidia_gpu", "has_amd_gpu",
    "Is_IPS", "Company_enc", "TypeName_enc", "OpSys_enc", "Cpu_Gamme_enc"
]
target = "Price"

X = df[features].fillna(0)
y = df[target]

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille train : {X_train.shape}, Taille test : {X_test.shape}")
print(f"Variable cible : Prix (min={y.min():.0f}, max={y.max():.0f}, médiane={y.median():.0f})")


### 3. Modélisation Vision / Deep Learning (CNN & TensorFlow)

Un **réseau de neurones convolutif (CNN)** est une architecture de deep learning conçue à la base pour traiter des images — mais elle peut aussi s'appliquer à des données tabulaires en les encodant sous forme de cartes de features.

**Architecture utilisée :**
- `Conv2D(16, 3×3)` — détecte des motifs locaux dans les features
- `MaxPooling2D(2×2)` — réduit la dimensionnalité
- `Flatten()` — aplatit en vecteur
- `Dense(32, relu)` + `Dropout(0.3)` — couche dense avec régularisation
- `Dense(1, sigmoid)` — sortie binaire : laptop "pas cher" vs "cher"

**Données d'entrée :** les 17 features tabulaires sont reshapées en cartes 5×5 (+ zero-padding).

> ⚠️ TensorFlow n'est pas encore compatible avec **Python 3.14 sur Windows ARM64**. Le code s'exécutera automatiquement si TensorFlow est disponible.

In [ ]:

# Entraînement du Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Modèle Random Forest entraîné avec succès !")
print(f"Nombre d'arbres : {rf_model.n_estimators}")


In [ ]:

# L'évaluation des métriques (MAE, RMSE, R²) est réalisée dans le notebook 06_evaluation
print("Modèle prêt — évaluation dans 06_evaluation.ipynb")


In [ ]:

# Importance des variables (analyse du modèle)
feat_imp = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
feat_imp.plot(kind='barh', figsize=(8, 6), color='steelblue')
plt.title("Importance des variables — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# CNN (Deep Learning) — classification du segment de prix d'un laptop
# NOTE : TensorFlow n'est pas encore compatible avec Python 3.14 (support prévu
#        pour TF 2.19+). Le code ci-dessous est fonctionnel et s'exécutera
#        automatiquement dès que TensorFlow sera disponible pour cette version.
# ──────────────────────────────────────────────────────────────────────────────

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    TENSORFLOW_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} disponible !")
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow non disponible sur Python 3.14 (incompatibilite connue).")
    print("Le code CNN ci-dessous illustre l'architecture prevue.\n")

if TENSORFLOW_AVAILABLE:
    # Encodage des features tabulaires en "images" 5x5 (padding zeros)
    import numpy as np_arr
    n_features = X_train.shape[1]  # 17 features
    pad = 25 - n_features          # padding pour atteindre 5x5=25
    X_img_train = np.pad(X_train.values, ((0, 0), (0, pad))).reshape(-1, 5, 5, 1).astype("float32")
    X_img_test  = np.pad(X_test.values,  ((0, 0), (0, pad))).reshape(-1, 5, 5, 1).astype("float32")

    # Labels : 0=pas cher (< médiane), 1=cher (>= médiane)
    mediane = y_train.median()
    y_img_train = (y_train.values >= mediane).astype(int)
    y_img_test  = (y_test.values  >= mediane).astype(int)

    cnn_model = models.Sequential([
        layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=(5, 5, 1)),
        layers.MaxPooling2D((2, 2), padding='same'),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn_model.summary()

    history = cnn_model.fit(X_img_train, y_img_train,
                            epochs=10, batch_size=32,
                            validation_split=0.1, verbose=1)
    loss, acc = cnn_model.evaluate(X_img_test, y_img_test, verbose=0)
    print(f"\nCNN - Precision sur le jeu de test : {acc:.2%}")

else:
    print("Architecture CNN (exécutable avec TensorFlow) :")
    print("  Conv2D(16, 3x3, relu, padding=same)  — input_shape=(5, 5, 1)")
    print("  MaxPooling2D(2x2, padding=same)")
    print("  Flatten()")
    print("  Dense(32, relu)  +  Dropout(0.3)")
    print("  Dense(1, sigmoid)  — sortie : P(laptop cher)")
    print("  Optimiseur : Adam  |  Loss : binary_crossentropy")
    print()
    print("Tache : classifier chaque laptop en 'pas cher' / 'cher' (seuil = mediane des prix)")
    print("Input : 17 features tabulaires reshapees en carte 5x5 (+ zero-padding)")
